# 🔥 Обновление данных сайта ГВС-Ярославль

Конвертирует Excel → `data.json`

In [ ]:
# ШАГ 0 — Установка библиотек
!pip install pandas openpyxl --quiet
print('✅ Готово')

In [ ]:
# ШАГ 1 — Загрузка Excel-файла
from google.colab import files
print('📂 Выберите .xlsx файл с таблицей отключений')
uploaded = files.upload()
xlsx_filename = list(uploaded.keys())[0]
print(f'\n✅ Загружен: {xlsx_filename}')

In [ ]:
# ШАГ 2 — Конвертация с нормализацией и дедупликацией
import pandas as pd
import json
import re

# ────────────────────────────────────────────────────────────────────────────
# Вспомогательные функции форматирования дат
# ────────────────────────────────────────────────────────────────────────────

def pad(x):
    """Дополняет число до двух цифр: 5 → '05'"""
    return f'{int(x):02d}'

def expand_year(yy_str):
    """Раскрывает двузначный год: '25' → '2025', '26' → '2026'.
    Четырёхзначные годы возвращаются без изменений."""
    y = int(yy_str)
    if y < 100:
        y += 2000 if y < 50 else 1900
    return str(y)

def format_date(val):
    """Форматирует одиночную дату в 'dd.mm.yyyy'.
    Принимает pandas Timestamp или строку."""
    if pd.isna(val) or val == '' or val is None:
        return ''
    if hasattr(val, 'strftime'):
        return val.strftime('%d.%m.%Y')
    return str(val).strip()

def format_period(val):
    """Нормализует строку периода отключения к виду 'dd.mm.yyyy[-dd.mm.yyyy]'.

    Поддерживаемые входные форматы (все из реальных Excel-файлов):
        '01.08.25-10.08.25'   → '01.08.2025-10.08.2025'
        '05.08.-18.08.25'     → '05.08.2025-18.08.2025'
        '08-11.07.25'         → '08.07.2025-11.07.2025'
        '25.07-04.08..2025'   → '25.07.2025-04.08.2025'
        '23-24.07.2025'       → '23.07.2025-24.07.2025'
        '28.12-03.01.26'      → '28.12.2025-03.01.2026'  (межгодовой)
        'По распоряжению ДГХ' → 'По распоряжению ДГХ'   (без изменений)
        'снесен'              → 'снесен'                 (без изменений)
    """
    if pd.isna(val) or val == '' or val is None:
        return ''
    if hasattr(val, 'strftime'):
        return val.strftime('%d.%m.%Y')

    s = str(val).strip()

    # Нормализация: двойные точки → одна, '.-' → '-', обрезаем крайние точки
    s = re.sub(r'\.{2,}', '.', s)
    s = re.sub(r'\.-', '-', s)
    s = s.strip('.')

    # 1. dd.mm.yyyy-dd.mm.yyyy  (нормализованный, 4-значный год)
    m = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{4})[-\u2013](\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if m:
        d1, mo1, y1, d2, mo2, y2 = m.groups()
        return f'{pad(d1)}.{pad(mo1)}.{y1}-{pad(d2)}.{pad(mo2)}.{y2}'

    # 2. dd.mm.yy-dd.mm.yy  (двузначный год у обеих частей)
    m = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{2})[-\u2013](\d{1,2})\.(\d{1,2})\.(\d{2})$', s)
    if m:
        d1, mo1, y1, d2, mo2, y2 = m.groups()
        return f'{pad(d1)}.{pad(mo1)}.{expand_year(y1)}-{pad(d2)}.{pad(mo2)}.{expand_year(y2)}'

    # 3. dd.mm-dd.mm.yy[yy]  (год только у второй части, 2 или 4 знака)
    m = re.match(r'^(\d{1,2})\.(\d{1,2})-(\d{1,2})\.(\d{1,2})\.(\d{2,4})$', s)
    if m:
        d1, mo1, d2, mo2, y = m.groups()
        y_end = expand_year(y)
        y_start = str(int(y_end) - 1) if int(mo1) > int(mo2) else y_end
        return f'{pad(d1)}.{pad(mo1)}.{y_start}-{pad(d2)}.{pad(mo2)}.{y_end}'

    # 4. dd-dd.mm.yy[yy]  (диапазон дней одного месяца)
    m = re.match(r'^(\d{1,2})-(\d{1,2})\.(\d{1,2})\.(\d{2,4})$', s)
    if m:
        d1, d2, mo, y = m.groups()
        return f'{pad(d1)}.{pad(mo)}.{expand_year(y)}-{pad(d2)}.{pad(mo)}.{expand_year(y)}'

    # 5. dd.mm.yyyy  (одиночная дата, 4-значный год)
    m = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if m:
        d, mo, y = m.groups()
        return f'{pad(d)}.{pad(mo)}.{y}'

    # 6. dd.mm.yy  (одиночная дата, двузначный год)
    m = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{2})$', s)
    if m:
        d, mo, yy = m.groups()
        return f'{pad(d)}.{pad(mo)}.{expand_year(yy)}'

    # Текстовые значения («По распоряжению…», «снесен», «ГВС нет» и т.д.)
    # Возвращаем как есть — фронтенд обрабатывает их отдельно.
    return s


# ────────────────────────────────────────────────────────────────────────────
# Нормализация адресов
# ────────────────────────────────────────────────────────────────────────────

STREET_SYNONYMS = [
    (r'\bпр-кт\.?(?=\s|,|$)',    'проспект'),
    (r'\bпр-т\.?(?=\s|,|$)',     'проспект'),
    (r'\bпроспект(?=\s|,|$)',    'проспект'),
    (r'\bпр-д\.?(?=\s|,|$)',     'проезд'),
    (r'\bпроезд(?=\s|,|$)',      'проезд'),
    (r'\bул\.?(?=\s|,|$)',       'ул.'),
    (r'\bулица(?=\s|,|$)',       'ул.'),
    (r'\bпер\.?(?=\s|,|$)',      'пер.'),
    (r'\bпереулок(?=\s|,|$)',    'пер.'),
    (r'\bбул\.?(?=\s|,|$)',      'бул.'),
    (r'\bбульвар(?=\s|,|$)',     'бул.'),
    (r'\bпл\.?(?=\s|,|$)',       'пл.'),
    (r'\bплощадь(?=\s|,|$)',     'пл.'),
    (r'\bст\.?(?=\s|,|$)',       'станция'),
    (r'\bнаб\.?(?=\s|,|$)',      'набережная'),
]

LOCALITY_SYNONYMS = [
    (r'\bп\.?(?=\s|,|$)',        'посёлок'),
    (r'\bпос\.?(?=\s|,|$)',      'посёлок'),
    (r'\bпосёлок(?=\s|,|$)',     'посёлок'),
    (r'\bпоселок(?=\s|,|$)',     'посёлок'),
]
LOCALITY_TYPES  = {'посёлок', 'станция', 'деревня', 'село', 'микрорайон'}
STREET_PREFIXES = {'проспект', 'проезд', 'ул.', 'пер.', 'бул.', 'пл.', 'шоссе', 'тупик', 'аллея'}

_raw = STREET_PREFIXES | LOCALITY_TYPES | {'набережная', 'д.', 'д'}
DEDUP_IGNORE = _raw | {re.sub(r'\.', '', t) for t in _raw}

def expand_bolshaya(s):
    """Б. Октябрьская / Б. ДОНСКАЯ → Большая Октябрьская / Большая Донская."""
    def repl(m):
        word = m.group(1)
        return 'Большая ' + word[0].upper() + word[1:].lower()
    s = re.sub(r'\bБ\.\s+([А-ЯЁа-яёA-Za-z]+)', repl, s)
    s = re.sub(r'\bБ\b\s+([А-ЯЁ]{2,})',         repl, s)
    return s

def normalize_address(addr):
    s = str(addr).strip()
    s = re.sub(r'^г\.?\s*Ярославль[,\s]+', '', s, flags=re.IGNORECASE)
    s = expand_bolshaya(s)
    s = re.sub(r'\bул\.([А-ЯЁа-яё])', r'ул. \1', s)
    for p, r in LOCALITY_SYNONYMS:
        s = re.sub(p, r, s, flags=re.IGNORECASE)
    for p, r in STREET_SYNONYMS:
        s = re.sub(p, r, s, flags=re.IGNORECASE)
    s = re.sub(r'ул\.([А-ЯЁа-яё])', r'ул. \1', s)
    s = re.sub(r'(?<=\s)Набережная\b', 'набережная', s)
    s = re.sub(r'\bд\.?\s*(\d)', r'д. \1', s)
    s = re.sub(r'\bкорп?\.?\s*(\d)', r'корп. \1', s)
    s = re.sub(r'(?<!\.)\s+(\d[\d/а-яА-ЯёЁ]*)$', r' д. \1', s)
    s = re.sub(r',{2,}', ',', s)
    s = re.sub(r'\s*,\s*', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    locality_re = '|'.join(re.escape(lt) for lt in LOCALITY_TYPES)
    m = re.match(
        r'^(ул\.)\s+(' + locality_re + r')\s+(\S+)\s+(.+?)(\s+д\..*)',
        s, flags=re.IGNORECASE
    )
    if m:
        s = f'{m.group(2)} {m.group(3)}, {m.group(1)} {m.group(4)}{m.group(5)}'
    tokens = s.split()
    first = tokens[0].lower().rstrip(',') if tokens else ''
    if first not in LOCALITY_TYPES:
        type_idx = next((i for i, t in enumerate(tokens) if t.lower() in STREET_PREFIXES), None)
        if type_idx is not None and type_idx != 0:
            w = tokens.pop(type_idx)
            tokens.insert(0, w)
            s = ' '.join(tokens)
    tokens, seen, clean = s.split(), False, []
    for t in tokens:
        if t.lower() in STREET_PREFIXES:
            if not seen: clean.append(t); seen = True
        else: clean.append(t)
    s = ' '.join(clean)
    prefixes_re = '|'.join(re.escape(p) for p in STREET_PREFIXES)
    s = re.sub(
        r'^(' + prefixes_re + r')\s+набережная\s+(.+?)(\s+д\..*)',
        r'\1 \2 набережная\3', s, flags=re.IGNORECASE
    )
    return s

def dedup_key(address_normalized, repair, hydro1, hydro2, source):
    """Ключ дедупликации: порядок слов и пунктуация не влияют на результат."""
    tokens = address_normalized.lower().split()
    tokens = [re.sub(r'[,.]', '', t) for t in tokens]
    tokens = [t for t in tokens if t and t not in DEDUP_IGNORE]
    return (' '.join(sorted(tokens)), repair, hydro1, hydro2, source)


# ────────────────────────────────────────────────────────────────────────────
# Основная функция конвертации
# ────────────────────────────────────────────────────────────────────────────

def convert(xlsx_path):
    xl = pd.ExcelFile(xlsx_path)
    raw_records = []

    for sheet_name in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet_name, header=None, skiprows=2)
        for _, row in df.iterrows():
            address_raw = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ''
            if not address_raw or address_raw == 'nan':
                continue

            source = str(row.iloc[10]).strip() if pd.notna(row.iloc[10]) else ''

            raw_records.append({
                'address':   normalize_address(address_raw),
                'repair':    format_period(row.iloc[4]),
                'repair_on': format_date(row.iloc[5]),
                'hydro1':    format_period(row.iloc[6]),
                'hydro1_on': format_date(row.iloc[7]),
                'hydro2':    format_period(row.iloc[8]),
                'hydro2_on': format_date(row.iloc[9]),
                'source':    source,
            })

    seen = set()
    unique = []
    for r in raw_records:
        key = dedup_key(r['address'], r['repair'], r['hydro1'], r['hydro2'], r['source'])
        if key not in seen:
            seen.add(key)
            unique.append(r)

    return raw_records, unique


print(f'⚙️  Обрабатываю: {xlsx_filename}\n')
raw, records = convert(xlsx_filename)

print(f'Исходных строк:     {len(raw)}')
print(f'После дедупликации: {len(records)}')
print(f'Удалено дублей:     {len(raw) - len(records)}')

# Диагностика: нестандартные значения периодов
import re as _re
_DATE_RE = _re.compile(r'^\d{2}\.\d{2}\.\d{4}(-\d{2}\.\d{2}\.\d{4})?$')
non_date = [
    (r['address'][:40], k, r[k])
    for r in records
    for k in ('repair', 'hydro1', 'hydro2')
    if r[k] and not _DATE_RE.match(r[k])
]
if non_date:
    print(f'\n⚠️  Нестандартные значения периодов ({len(non_date)} шт.) — фронтенд покажет их как заметки:')
    for addr, field, val in non_date[:10]:
        print(f'  [{field}] {val!r:35s} | {addr}')
    if len(non_date) > 10:
        print(f'  ... и ещё {len(non_date) - 10}')

with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f'\n✅ Сохранено: data.json')

In [ ]:
# ШАГ 3 — Верификация: проверяем несколько адресов вручную
import json

with open('data.json', encoding='utf-8') as f:
    data = json.load(f)

print(f'Всего записей: {len(data)}')
print(f'Первые 3 записи:\n')
for d in data[:3]:
    print(json.dumps(d, ensure_ascii=False, indent=2))
    print()

In [ ]:
# ШАГ 4 — Скачать data.json
from google.colab import files
files.download('data.json')
print('⬇️  Скачан data.json')